<!-- dads-lab-header -->
<a href="https://colab.research.google.com/github/mdehghani86/DADS5250-GenAI/blob/main/labs/M03/M03_Lab2_JSON_Mode_Pydantic.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

<div style="background: linear-gradient(135deg, #001a70 0%, #0055d4 100%); padding: 30px 36px; border-radius: 14px; margin-bottom: 20px;">
  <h1 style="color: #fff; margin: 0; font-size: 28px;">🧱 M03 Lab 2 — JSON Mode & Pydantic</h1>
  <p style="color: rgba(255,255,255,0.85); margin: 8px 0 0; font-size: 15px;">DADS 5250: Generative AI in Practice &nbsp;|&nbsp; Prof. Dehghani</p>
  <p style="color: rgba(255,255,255,0.65); margin: 6px 0 0; font-size: 13px;">⭐ Difficulty: Intermediate &nbsp;|&nbsp; ⏱ Time: ~30 min</p>
</div>

> **📌 Note on models.** This lab references specific LLM versions through `DEFAULT_CHAT_MODEL` and `DEFAULT_MINI_MODEL` constants in the `dads5250` utility package. Models update quickly — feel free to swap to any newer OpenAI / Anthropic / Google model you have access to.


In [ ]:
# === Shared lab setup: install dads5250 + load API key + sticky pill ===
# Installs the shared utilities (pp, pretty_print, lab_pill, model constants,
# setup_openai, setup_gemini) once per Colab runtime. The same OPENAI_API_KEY
# / GEMINI_API_KEY Colab secrets are used across every DADS 5250 lab — set
# them once in the 🔑 sidebar and they're picked up automatically.
import os
import importlib.util
if importlib.util.find_spec("dads5250") is None:
    !pip install -q "git+https://github.com/mdehghani86/DADS5250-GenAI.git#subdirectory=utils"

from dads5250 import (
    pp,
    pretty_print,
    lab_pill,
    setup_openai,
    setup_gemini,
    DEFAULT_CHAT_MODEL,   # newest reasoning model that supports temperature
    DEFAULT_MINI_MODEL,   # newest mini model that supports temperature
    DEFAULT_EMBED_MODEL,  # current embeddings default
    DEFAULT_GEMINI_MODEL, # tracks the latest stable flash
)

lab_pill('M03 Lab 2 — JSON Mode & Pydantic')            # sticky banner so you always see which lab you're in
client = setup_openai()        # loads OPENAI_API_KEY + verifies it works


In [ ]:
# ============================================================
# 🔑 Setup
# ============================================================
import json
from pydantic import BaseModel
from openai import OpenAI
from dads5250 import setup_openai, pp, show_response, show_expected, show_info, quiz

client = setup_openai()

<div style="background: linear-gradient(135deg, #001a70 0%, #0055d4 100%); padding: 18px 24px; border-radius: 10px; margin: 20px 0 12px;">
  <h2 style="color: white; margin: 0; font-size: 20px;">1️⃣ JSON Mode</h2>
</div>

By default, LLMs return free-form text. **JSON mode** forces the model to return **valid JSON** — guaranteed parseable by `json.loads()`.

Two requirements:
1. Set `response_format={"type": "json_object"}`
2. Include the word "JSON" in your system or user message (OpenAI requires this)

In [ ]:
# ============================================================
# 📋 JSON Mode — Basic Example
# ============================================================

response = client.chat.completions.create(
    model="gpt-4.1-mini",
    messages=[
        {"role": "system", "content": "You are a helpful assistant. Always respond in JSON format."},
        {"role": "user", "content": "List 3 popular programming languages with their primary use case and year created."}
    ],
    response_format={"type": "json_object"},
)

# ✅ This is guaranteed to be valid JSON
data = json.loads(response.choices[0].message.content)
pp(data, "JSON Mode Response — Programming Languages")

<div style="background: #fff8e1; border-left: 4px solid #f9a825; padding: 12px 16px; border-radius: 0 8px 8px 0; margin: 12px 0;">
  <b>💡 Pro Tip:</b> Without JSON mode, the model <i>might</i> return JSON if you ask nicely, but it could also wrap it in markdown code blocks, add commentary, or return malformed JSON. JSON mode <b>guarantees</b> valid JSON every time — no more brittle regex parsing.
</div>

<div style="background: linear-gradient(135deg, #001a70 0%, #0055d4 100%); padding: 18px 24px; border-radius: 10px; margin: 20px 0 12px;">
  <h2 style="color: white; margin: 0; font-size: 20px;">2️⃣ Real-World Extraction: Invoice Parser</h2>
</div>

One of the most common enterprise LLM use cases is **extracting structured data from unstructured documents**. Let's build an invoice parser — imagine processing hundreds of invoices automatically.

In [ ]:
# ============================================================
# 🧾 Invoice Text → Structured JSON
# ============================================================

invoice_text = """
Invoice #INV-2024-0847
Date: March 15, 2024
Bill To: Acme Corp, 123 Main St, Boston MA 02101

Items:
- Cloud hosting (monthly): $450.00
- API usage (10K calls): $125.50
- Support plan (premium): $200.00

Subtotal: $775.50
Tax (6.25%): $48.47
Total Due: $823.97
Payment Terms: Net 30
"""

extraction_prompt = """Extract structured data from this invoice. Return JSON with these exact keys:
invoice_number, date, customer, items (list of {name, amount}), subtotal, tax, total"""

response = client.chat.completions.create(
    model="gpt-4.1-mini",
    messages=[
        {"role": "system", "content": extraction_prompt},
        {"role": "user", "content": invoice_text}
    ],
    response_format={"type": "json_object"},
)

invoice_data = json.loads(response.choices[0].message.content)
pp(invoice_data, "Extracted Invoice Data")

<div style="background: #f0f4ff; border-left: 4px solid #0055d4; padding: 14px 18px; border-radius: 0 8px 8px 0; margin: 12px 0; font-size: 14px;">
  <b>🧠 Notice:</b> The model correctly identified the invoice number, parsed dollar amounts as numbers, separated line items into a list, and calculated tax. This would have taken dozens of regex patterns to do programmatically — the LLM handles it in one call.
</div>

<div style="background: linear-gradient(135deg, #001a70 0%, #0055d4 100%); padding: 18px 24px; border-radius: 10px; margin: 20px 0 12px;">
  <h2 style="color: white; margin: 0; font-size: 20px;">3️⃣ Pydantic Validation</h2>
</div>

JSON mode guarantees *valid JSON*, but not the *right structure*. What if the model uses `"total_amount"` instead of `"total"`? Or returns a string `"$823.97"` instead of the number `823.97`?

**Pydantic** solves this — you define a Python data model, and it validates the LLM output against your expected schema. If anything is wrong, you get a clear error instead of a silent bug downstream.

In [ ]:
# ============================================================
# ✅ Define Pydantic Models
# ============================================================

class InvoiceItem(BaseModel):
    name: str
    amount: float

class Invoice(BaseModel):
    invoice_number: str
    date: str
    customer: str
    items: list[InvoiceItem]
    subtotal: float
    tax: float
    total: float

# Validate the extracted data
try:
    validated = Invoice(**invoice_data)
    print("✅ Validation passed!\n")
    print(f"📋 Invoice:  {validated.invoice_number}")
    print(f"👤 Customer: {validated.customer}")
    print(f"📅 Date:     {validated.date}")
    print(f"\n📦 Items ({len(validated.items)}):")
    for item in validated.items:
        print(f"   - {item.name}: ${item.amount:.2f}")
    print(f"\n💰 Subtotal: ${validated.subtotal:.2f}")
    print(f"📊 Tax:      ${validated.tax:.2f}")
    print(f"💵 Total:    ${validated.total:.2f}")
except Exception as e:
    print(f"❌ Validation failed: {e}")

<div style="background: #fff8e1; border-left: 4px solid #f9a825; padding: 12px 16px; border-radius: 0 8px 8px 0; margin: 12px 0;">
  <b>💡 Why Pydantic Matters in Production:</b> Imagine your invoice extractor runs 1,000 times a day. Without validation, a single malformed response could crash your pipeline, corrupt your database, or send a wrong amount to accounting. Pydantic catches these errors <i>before</i> they cause damage.
</div>

<div style="background: linear-gradient(135deg, #001a70 0%, #0055d4 100%); padding: 18px 24px; border-radius: 10px; margin: 20px 0 12px;">
  <h2 style="color: white; margin: 0; font-size: 20px;">4️⃣ Extraction at Scale: Multiple Document Types</h2>
</div>

The same pattern works for any document type. Let's test it on three completely different inputs and see how well the model adapts:

In [ ]:
# ============================================================
# 📄 Multi-Document Extraction
# ============================================================

documents = {
    "Restaurant Menu": """
BELLA'S ITALIAN - LUNCH MENU
Margherita Pizza - Fresh mozzarella, basil, tomato sauce - $14.95
Caesar Salad - Romaine, parmesan, croutons, house dressing - $11.50
Penne Arrabiata - Spicy tomato sauce, garlic, chili flakes - $16.00
Tiramisu - Classic Italian dessert - $8.50
    """,

    "Job Posting": """
SENIOR ML ENGINEER - TechCorp (Remote)
Salary: $150,000 - $200,000
Requirements: 5+ years Python, PyTorch/TensorFlow, cloud deployment (AWS/GCP)
Nice to have: LLM fine-tuning experience, Kubernetes
Benefits: Health, dental, 401k match, unlimited PTO
Apply by: April 30, 2024
    """,

    "Product Review": """
Samsung Galaxy S24 Ultra Review - 4.5/5 stars
Pros: Amazing camera (200MP), great battery life (2 days), S-Pen is useful
Cons: Expensive ($1,299), heavy (233g), AI features feel gimmicky
Verdict: Best Android phone if budget isn't a concern.
    """
}

system = "Extract ALL structured data from this document. Return well-organized JSON."

for doc_type, text in documents.items():
    r = client.chat.completions.create(
        model="gpt-4.1-mini",
        messages=[
            {"role": "system", "content": system},
            {"role": "user", "content": text}
        ],
        response_format={"type": "json_object"},
    )
    parsed = json.loads(r.choices[0].message.content)
    pp(parsed, f"Extracted — {doc_type}")

**📝 Your Observations:** *(double-click to edit)*

| Document Type | Extraction Quality | Any Missing/Wrong Fields? | Notes |
|:---:|---|---|---|
| Restaurant Menu | _[Excellent / Good / Fair]_ | _[Yes/No — describe]_ | |
| Job Posting | _[Excellent / Good / Fair]_ | _[Yes/No — describe]_ | |
| Product Review | _[Excellent / Good / Fair]_ | _[Yes/No — describe]_ | |

<div style="background: linear-gradient(135deg, #001a70 0%, #0055d4 100%); padding: 18px 24px; border-radius: 10px; margin: 20px 0 12px;">
  <h2 style="color: white; margin: 0; font-size: 20px;">📝 Knowledge Check</h2>
</div>

In [ ]:
quiz([
    {
        "q": "What is the key difference between JSON mode and function calling?",
        "options": [
            "JSON mode is faster than function calling",
            "JSON mode returns free-form JSON; function calling returns structured calls matching a predefined schema",
            "Function calling only works with GPT-4",
            "There is no practical difference"
        ],
        "answer": 1,
        "explanation": "JSON mode guarantees valid JSON but the structure is flexible — the model decides the keys. Function calling defines exact schemas the model must follow, returning structured function invocations with specific arguments."
    }
])

<div style="background: linear-gradient(135deg, #001a70 0%, #0055d4 100%); padding: 18px 24px; border-radius: 10px; margin: 20px 0 12px;">
  <h2 style="color: white; margin: 0; font-size: 20px;">🔧 Hands-On Exercise: Build Your Own Extractor</h2>
</div>

Extract structured data from a **real estate listing**. Complete the system prompt and define a Pydantic model.

In [ ]:
# ============================================================
# 🔧 Exercise: Real Estate Listing Extractor
# ============================================================

listing = """
Charming 3BR/2BA Colonial in Brookline, MA — $875,000
Built in 1928, recently renovated. 1,850 sq ft on a quiet tree-lined street.
Features: hardwood floors, updated kitchen with granite counters, 
finished basement, central AC, 2-car garage.
Walk to Coolidge Corner shops and Green Line T stop.
Open house: Saturday, March 23, 1-3pm
Agent: Sarah Chen, Compass Realty — (617) 555-0142
"""

# Step 1: Write the extraction prompt
# Replace ----- with your system prompt
extraction_system = "-----"
# 👆 YOUR CODE: Tell the model what fields to extract
#    (e.g., bedrooms, bathrooms, price, sqft, features, agent, etc.)

r = client.chat.completions.create(
    model="gpt-4.1-mini",
    messages=[
        {"role": "system", "content": extraction_system},
        {"role": "user", "content": listing}
    ],
    response_format={"type": "json_object"},
)

listing_data = json.loads(r.choices[0].message.content)
print("🏠 Extracted Listing:")
print(json.dumps(listing_data, indent=2))

In [ ]:
# ============================================================
# 🔧 Exercise (continued): Validate with Pydantic
# ============================================================

# Step 2: Define a Pydantic model for the listing
# Replace ----- with the correct field types

class RealEstateListing(BaseModel):
    bedrooms: -----          # What type?
    bathrooms: -----         # What type?
    price: -----             # What type?
    sqft: -----              # What type?
    city: -----              # What type?
    features: -----          # What type? (hint: it's a list)

# Step 3: Validate
try:
    validated_listing = RealEstateListing(**listing_data)
    print(f"✅ Valid! {validated_listing.bedrooms}BR in {validated_listing.city} — ${validated_listing.price:,}")
except Exception as e:
    print(f"❌ Validation error: {e}")
    print("\n💡 Hint: Check if the JSON keys match your Pydantic field names.")
    print("   You may need to adjust either the prompt or the model.")

<div style="background: #e8f5e9; border-left: 4px solid #4caf50; padding: 14px 18px; border-radius: 0 8px 8px 0; margin: 12px 0;">
  <h3 style="margin: 0 0 8px;">📖 Self-Study Activity</h3>
  <ol style="font-size: 14px;">
    <li>Add <b>optional fields</b> to your Pydantic model (e.g., <code>garage: int | None = None</code>)</li>
    <li>Try extracting from a listing that's <b>missing some info</b> — how does the model handle it?</li>
    <li>Add <b>nested models</b> (e.g., <code>class Agent(BaseModel): name, phone, company</code>)</li>
  </ol>
</div>

<div style="background: linear-gradient(135deg, #001a70 0%, #0055d4 100%); padding: 24px 30px; border-radius: 12px; margin: 24px 0 0; text-align: center;">
  <h2 style="color: white; margin: 0 0 10px;">🎉 Lab 1 Complete!</h2>
  <p style="color: rgba(255,255,255,0.85); margin: 0; font-size: 14px;">You can now force structured output from LLMs and validate it programmatically.</p>
  <div style="background: rgba(255,255,255,0.15); border-radius: 8px; padding: 12px; margin-top: 14px; text-align: left;">
    <p style="color: white; margin: 0 0 6px; font-weight: bold; font-size: 14px;">Key Takeaways:</p>
    <ul style="color: rgba(255,255,255,0.9); margin: 0; font-size: 13px;">
      <li><code style="color: #90caf9;">response_format={"type": "json_object"}</code> guarantees valid JSON</li>
      <li>Always include "JSON" in your system/user message when using JSON mode</li>
      <li><b>Pydantic</b> validates structure, types, and required fields — essential for production</li>
      <li>The same extraction pattern works for invoices, menus, reviews, listings — any document</li>
    </ul>
  </div>
  <p style="color: rgba(255,255,255,0.7); margin: 14px 0 0; font-size: 13px;"><b>Next →</b> M03 Lab 2: Function Calling &amp; Tool Use</p>
</div>